# 01a -- dim_rookie_prospect Seed

**Purpose**: Build the three pre-draft seed tables:
- `dim_position` and `dim_school` — transformer tables used by all ETL
- `dim_rookie_prospect` — current draft class prospect registry

`dim_rookie_prospect` is a **staging dimension**. Prospects graduate to
`dim_nfl_players` (the central player registry) once they sign their rookie
contracts post-draft and receive a `gsis_id` from the NFL.

**Key**: `player_key` (interim MD5 hash, pre-`gsis_id`)

**Outputs** (local Parquet):
- `data/dim_position.parquet`
- `data/dim_school.parquet`
- `data/dim_rookie_prospect.parquet`

In [1]:
# ---- Setup & Config (shared module) ----------------------------------------
# All config + path anchoring + helpers live in notebooks/etl_helpers.py.
# CFG.data_dir / DATA / REVIEW are anchored to the repo root -> writes always
# land in <repo>/data no matter the CWD this notebook runs from.
import sys
from pathlib import Path
for _p in (Path.cwd() / "notebooks", Path.cwd(), Path.cwd().parent):
    if (_p / "etl_helpers.py").exists():
        sys.path.insert(0, str(_p)); break
import etl_helpers as etl
from etl_helpers import (LeagueConfig, CFG, DATA, REVIEW, TODAY, ALIAS,
                         clean_player_name, generate_player_key, parse_height_to_inches)

# !pip install requests
# !pip install nflreadpy

import pandas as pd
import numpy as np
import hashlib
import nflreadpy as nfl
from pathlib import Path
from datetime import datetime, date
from dataclasses import dataclass, field

from thefuzz import fuzz, process


## 1 — dim_Position (transformer table)

Maps every raw position string you'll encounter across sources to a 
canonical `position_detail`, a `position_group` for filtering, and an
`offense_defense` flag. This replaces the giant if/else chains from 
the old M code.

**Maintenance**: When a new raw position appears from a source, add a row.

In [2]:
# ── Position transformer ────────────────────────────────────────────────────
# Columns:
#   position_raw    : exactly as it appears in the source (case-insensitive match)
#   position_detail : the granular canonical name (e.g., EDGE, CB, IOL)
#   position_group  : rolled-up group for fantasy filtering (QB, RB, WR, TE, DL, LB, DB, OL, ST)
#   side_of_ball    : Offense / Defense / Special Teams
#   side_of_ball_sort_order: 1=Offense, 2=Defense, 3=Special Teams
#   fantasy_relevant: True for positions that score in dynasty leagues

_pos_rows = [
    # ── Offense: Skill ──────────────────────────────────────────────────────
    ("QB",   "QB",   "QB", 1, "Offense", 1, True),
    ("RB",   "RB",   "RB", 2, "Offense", 1, True),
    ("HB",   "RB",   "RB", 2, "Offense", 1, True),   # CBS uses HB
    ("FB",   "FB",   "RB", 2, "Offense", 1, False),
    ("WR",   "WR",   "WR", 3, "Offense", 1, True),
    ("TE",   "TE",   "TE", 4, "Offense", 1, True),

    # ── Offense: Line ───────────────────────────────────────────────────────
    ("OT",   "OT",   "OL", 999, "Offense", 1, False),
    ("OG",   "OG",   "OL", 999, "Offense", 1, False),
    ("OL",   "OL",   "OL", 999, "Offense", 1, False),   # generic
    ("C",    "C",    "OL", 999, "Offense", 1, False),
    ("G",    "OG",   "OL", 999, "Offense", 1, False),   # ESPN uses bare G
    ("T",    "OT",   "OL", 999, "Offense", 1, False),   # ESPN uses bare T
    ("IOL",  "IOL",  "OL", 999, "Offense", 1, False),
    ("OC",   "C",    "OL", 999, "Offense", 1, False),   # some sources
    ("TG",   "OG",   "OL", 999, "Offense", 1, False),   # tackle/guard tweener
    ("GT",   "OG",   "OL", 999, "Offense", 1, False),

    # ── Defense: Line ───────────────────────────────────────────────────────
    ("EDGE", "EDGE", "DL", 6,  "Defense", 2, True),
    ("ED",   "EDGE", "DL", 6,  "Defense", 2, True),   # PFF uses ED
    ("DE",   "DE",   "DL", 6,  "Defense", 2, True),
    ("DT",   "DT",   "DL", 6,  "Defense", 2, True),
    ("DI",   "DI",   "DL", 6,  "Defense", 2, True),   # PFF interior
    ("DL",   "DL",   "DL", 6,  "Defense", 2, True),   # generic
    ("NT",   "NT",   "DL", 6,  "Defense", 2, True),

    # ── Defense: Linebacker ─────────────────────────────────────────────────
    ("LB",   "LB",   "LB", 7,  "Defense", 2, True),
    ("ILB",  "ILB",  "LB", 7,  "Defense", 2, True),
    ("OLB",  "OLB",  "LB", 7,  "Defense", 2, True),
    ("MLB",  "ILB",  "LB", 7,  "Defense", 2, True),
    ("WILL", "OLB",  "LB", 7,  "Defense", 2, True),
    ("SAM",  "OLB",  "LB", 7,  "Defense", 2, True),
    ("MIKE", "ILB",  "LB", 7,  "Defense", 2, True),

    # ── Defense: Secondary ──────────────────────────────────────────────────
    ("CB",   "CB",   "DB", 8,  "Defense", 2, True),
    ("S",    "S",    "DB", 8,  "Defense", 2, True),
    ("SAF",  "S",    "DB", 8,  "Defense", 2, True),
    ("FS",   "FS",   "DB", 8,  "Defense", 2, True),
    ("SS",   "SS",   "DB", 8,  "Defense", 2, True),
    ("DB",   "DB",   "DB", 8,  "Defense", 2, True),   # generic

    # ── Special Teams ───────────────────────────────────────────────────────
    ("K",    "K",    "ST", 999,  "Special Teams", 3, False),
    ("P",    "P",    "ST", 999,  "Special Teams", 3, False),
    ("LS",   "LS",   "ST", 999,  "Special Teams", 3, False),
]

dim_position = pd.DataFrame(
    _pos_rows,
    columns=["position_raw", "position_detail",
             "position_group", "position_sort_order",
             "side_of_ball", "side_of_ball_sort_order", "fantasy_relevant"]
)

dim_position["position_raw"] = dim_position["position_raw"].str.upper().str.strip()

out_path = Path(CFG.data_dir) / "dim_position.parquet"
dim_position.to_parquet(out_path, index=False)
print(f"dim_position: {len(dim_position)} mappings → {out_path}")
dim_position.head(10)

dim_position: 39 mappings → C:\Users\benha\OneDrive\Documents\GitHub\Python-PowerBI-DynastyFantasyFootball\data\dim_position.parquet


,position_raw,position_detail,position_group,position_sort_order,side_of_ball,side_of_ball_sort_order,fantasy_relevant
0,QB,QB,QB,1,Offense,1,True
1,RB,RB,RB,2,Offense,1,True
2,HB,RB,RB,2,Offense,1,True
3,FB,FB,RB,2,Offense,1,False
4,WR,WR,WR,3,Offense,1,True
5,TE,TE,TE,4,Offense,1,True
6,OT,OT,OL,999,Offense,1,False
7,OG,OG,OL,999,Offense,1,False
8,OL,OL,OL,999,Offense,1,False
9,C,C,OL,999,Offense,1,False


## 2 — dim_School (transformer table)

Maps raw school names across sources to a canonical name and conference.
Seeded from nflverse combine data, then extended as new names appear.

**Maintenance**: When fuzzy-match flags an unknown school, add a row.

In [4]:
# ── Comprehensive School Aliases ──────────────────────────────────────────
from pickle import INST


_school_aliases = {
    # ── AAC ──────────────────────────────────────────────────────────────
    "Army":                    ("Army", "ARMY", "AAC"),
    "Army Black Knights":      ("Army", "ARMY", "AAC"),
    "Charlotte":               ("Charlotte", "CLT", "AAC"),
    "Charlotte 49ers":         ("Charlotte", "CLT", "AAC"),
    "CLT":               ("Charlotte", "CLT", "AAC"),
    "ECU":                     ("East Carolina", "ECU", "AAC"),
    "East Carolina":           ("East Carolina", "ECU", "AAC"),
    "East Carolina Pirates":   ("East Carolina", "ECU", "AAC"),
    "FAU":                     ("Florida Atlantic", "FAU", "AAC"),
    "Florida Atlantic":        ("Florida Atlantic", "FAU", "AAC"),
    "Florida Atlantic Owls":   ("Florida Atlantic", "FAU", "AAC"),
    "Memphis":                 ("Memphis", "MEM", "AAC"),
    "Memphis Tigers":          ("Memphis", "MEM", "AAC"),
    "Navy":                    ("Navy", "NAVY", "AAC"),
    "Navy Midshipmen":         ("Navy", "NAVY", "AAC"),
    "Rice":                    ("Rice", "RICE", "AAC"),
    "Rice Owls":               ("Rice", "RICE", "AAC"),
    "Temple":                  ("Temple", "TEM", "AAC"),
    "Temple Owls":             ("Temple", "TEM", "AAC"),
    "TLSA":                    ("Tulsa", "TLSA", "AAC"),
    "Tulsa":                   ("Tulsa", "TLSA", "AAC"),
    "Tulsa Golden Hurricane":  ("Tulsa", "TLSA", "AAC"),
    "Tulane":                  ("Tulane", "TULN", "AAC"),
    "Tulane Green Wave":       ("Tulane", "TULN", "AAC"),
    "UAB":                     ("UAB", "UAB", "AAC"),
    "UAB Blazers":             ("UAB", "UAB", "AAC"),
    "North Texas":             ("North Texas", "UNT", "AAC"),
    "North Texas Mean Green":  ("North Texas", "UNT", "AAC"),
    "UNT":                     ("North Texas", "UNT", "AAC"),
    "South Florida":           ("South Florida", "USF", "AAC"),
    "South Florida Bulls":     ("South Florida", "USF", "AAC"),
    "USF":                     ("South Florida", "USF", "AAC"),
    "UT San Antonio":          ("UT San Antonio", "UTSA", "AAC"),
    "UTSA":                    ("UT San Antonio", "UTSA", "AAC"),
    "UTSA Roadrunners":        ("UT San Antonio", "UTSA", "AAC"),

    # ── ACC ──────────────────────────────────────────────────────────────
    "BC":                      ("Boston College", "BC", "ACC"),
    "Boston College":          ("Boston College", "BC", "ACC"),
    "Boston College Eagles":   ("Boston College", "BC", "ACC"),
    "Cal":                     ("California", "CAL", "ACC"),
    "California":              ("California", "CAL", "ACC"),
    "California Golden Bears": ("California", "CAL", "ACC"),
    "Clem":                    ("Clemson", "CLEM", "ACC"),
    "Clemson":                 ("Clemson", "CLEM", "ACC"),
    "Clemson Tigers":          ("Clemson", "CLEM", "ACC"),
    "Duke":                    ("Duke", "DUKE", "ACC"),
    "Duke Blue Devils":        ("Duke", "DUKE", "ACC"),
    "FSU":                     ("Florida State", "FSU", "ACC"),
    "Florida St":              ("Florida State", "FSU", "ACC"),
    "Florida St.":             ("Florida State", "FSU", "ACC"),
    "Florida State":           ("Florida State", "FSU", "ACC"),
    "Florida State Seminoles": ("Florida State", "FSU", "ACC"),
    "GT":                      ("Georgia Tech", "GT", "ACC"),
    "Georgia Tech":            ("Georgia Tech", "GT", "ACC"),
    "Georgia Tech Yellow Jackets": ("Georgia Tech", "GT", "ACC"),
    "Louisville":              ("Louisville", "LOU", "ACC"),
    "Louisville Cardinals":    ("Louisville", "LOU", "ACC"),
    "UofL":                    ("Louisville", "LOU", "ACC"),
    "Miami":                   ("Miami (FL)", "MIA", "ACC"),
    "Miami (FL)":              ("Miami (FL)", "MIA", "ACC"),
    "Miami FL":                ("Miami (FL)", "MIA", "ACC"),
    "Miami Hurricanes":        ("Miami (FL)", "MIA", "ACC"),
    "Miami-FL":                ("Miami (FL)", "MIA", "ACC"),
    "N.C. State":              ("North Carolina State", "NCST", "ACC"),
    "NC St":                   ("North Carolina State", "NCST", "ACC"),
    "NC State":                ("North Carolina State", "NCST", "ACC"),
    "NC State Wolfpack":       ("North Carolina State", "NCST", "ACC"),
    "North Carolina State":    ("North Carolina State", "NCST", "ACC"),
    "Pitt":                    ("Pittsburgh", "PITT", "ACC"),
    "Pittsburgh":              ("Pittsburgh", "PITT", "ACC"),
    "Pittsburgh Panthers":     ("Pittsburgh", "PITT", "ACC"),
    "SMU":                     ("Southern Methodist", "SMU", "ACC"),
    "SMU Mustangs":            ("Southern Methodist", "SMU", "ACC"),
    "Southern Methodist":      ("Southern Methodist", "SMU", "ACC"),
    "Stanford":                ("Stanford", "STAN", "ACC"),
    "Stanford Cardinal":       ("Stanford", "STAN", "ACC"),
    "Cuse":                    ("Syracuse", "SYR", "ACC"),
    "Syracuse":                ("Syracuse", "SYR", "ACC"),
    "Syracuse Orange":         ("Syracuse", "SYR", "ACC"),
    "N.C. Tarheels":           ("North Carolina", "UNC", "ACC"),
    "North Carolina":          ("North Carolina", "UNC", "ACC"),
    "North Carolina Tar Heels": ("North Carolina", "UNC", "ACC"),
    "UNC":                     ("North Carolina", "UNC", "ACC"),
    "UVA":                     ("Virginia", "UVA", "ACC"),
    "Virginia":                ("Virginia", "UVA", "ACC"),
    "Virginia Cavaliers":      ("Virginia", "UVA", "ACC"),
    "VT":                      ("Virginia Tech", "VT", "ACC"),
    "VTech":                   ("Virginia Tech", "VT", "ACC"),
    "Virginia Tech":           ("Virginia Tech", "VT", "ACC"),
    "Virginia Tech Hokies":    ("Virginia Tech", "VT", "ACC"),
    "Wake":                    ("Wake Forest", "WAKE", "ACC"),
    "Wake Forest":             ("Wake Forest", "WAKE", "ACC"),
    "Wake Forest Demon Deacons": ("Wake Forest", "WAKE", "ACC"),

    # ── Big 12 ───────────────────────────────────────────────────────────
    "Arizona":                 ("Arizona", "ARIZ", "Big 12"),
    "ASU":                     ("Arizona State", "ASU", "Big 12"),
    "Arizona State":           ("Arizona State", "ASU", "Big 12"),
    "Baylor":                  ("Baylor", "BAY", "Big 12"),
    "BYU":                     ("Brigham Young", "BYU", "Big 12"),
    "Brigham Young":           ("Brigham Young", "BYU", "Big 12"),
    "Cincinnati":              ("Cincinnati", "CIN", "Big 12"),
    "Cincy":                   ("Cincinnati", "CIN", "Big 12"),
    "Colorado":                ("Colorado", "COLO", "Big 12"),
    "Colorado Buffaloes":      ("Colorado", "COLO", "Big 12"),
    "Houston":                 ("Houston", "HOU", "Big 12"),
    "UH":                      ("Houston", "HOU", "Big 12"),
    "ISU":                     ("Iowa State", "ISU", "Big 12"),
    "Iowa St":                 ("Iowa State", "ISU", "Big 12"),
    "Iowa St.":                ("Iowa State", "ISU", "Big 12"),
    "Iowa State":              ("Iowa State", "ISU", "Big 12"),
    "KU":                      ("Kansas", "KAN", "Big 12"),
    "Kansas":                  ("Kansas", "KAN", "Big 12"),
    "Kansas Jayhawks":         ("Kansas", "KAN", "Big 12"),
    "K-State":                 ("Kansas State", "KSU", "Big 12"),
    "KSU":                     ("Kansas State", "KSU", "Big 12"),
    "Kansas St.":              ("Kansas State", "KSU", "Big 12"),
    "Kansas State":            ("Kansas State", "KSU", "Big 12"),
    "OKST":                    ("Oklahoma State", "OKST", "Big 12"),
    "Ok State":                ("Oklahoma State", "OKST", "Big 12"),
    "Oklahoma St.":            ("Oklahoma State", "OKST", "Big 12"),
    "Oklahoma State":          ("Oklahoma State", "OKST", "Big 12"),
    "Oklahoma State Cowboys":  ("Oklahoma State", "OKST", "Big 12"),
    "TCU":                     ("Texas Christian", "TCU", "Big 12"),
    "Texas Christian":         ("Texas Christian", "TCU", "Big 12"),
    "TTU":                     ("Texas Tech", "TTU", "Big 12"),
    "Texas Tech":              ("Texas Tech", "TTU", "Big 12"),
    "Central Florida":         ("Central Florida", "UCF", "Big 12"),
    "UCF":                     ("Central Florida", "UCF", "Big 12"),
    "UCF Knights":             ("Central Florida", "UCF", "Big 12"),
    "Utah":                    ("Utah", "UTAH", "Big 12"),
    "WVU":                     ("West Virginia", "WVU", "Big 12"),
    "West Virginia":           ("West Virginia", "WVU", "Big 12"),
    "West Virginia Mountaineers": ("West Virginia", "WVU", "Big 12"),

    # ── Big Ten ──────────────────────────────────────────────────────────
    "Ill":                     ("Illinois", "ILL", "Big Ten"),
    "Illinois":                ("Illinois", "ILL", "Big Ten"),
    "Illinois Fighting Illini": ("Illinois", "ILL", "Big Ten"),
    "IU":                      ("Indiana", "IND", "Big Ten"),
    "Indiana":                 ("Indiana", "IND", "Big Ten"),
    "Indiana Hoosiers":        ("Indiana", "IND", "Big Ten"),
    "Hawkeyes":                ("Iowa", "IOWA", "Big Ten"),
    "Iowa":                    ("Iowa", "IOWA", "Big Ten"),
    "Iowa Hawkeyes":           ("Iowa", "IOWA", "Big Ten"),
    "MD":                      ("Maryland", "MD", "Big Ten"),
    "Maryland":                ("Maryland", "MD", "Big Ten"),
    "Maryland Terrapins":      ("Maryland", "MD", "Big Ten"),
    "Mich.":                   ("Michigan", "MICH", "Big Ten"),
    "Michigan":                ("Michigan", "MICH", "Big Ten"),
    "Michigan Wolverines":     ("Michigan", "MICH", "Big Ten"),
    "Minn":                    ("Minnesota", "MINN", "Big Ten"),
    "Minnesota":               ("Minnesota", "MINN", "Big Ten"),
    "Minnesota Golden Gophers": ("Minnesota", "MINN", "Big Ten"),
    "MSU":                     ("Michigan State", "MSU", "Big Ten"),
    "Mich St":                 ("Michigan State", "MSU", "Big Ten"),
    "Mich St.":                ("Michigan State", "MSU", "Big Ten"),
    "Michigan State":          ("Michigan State", "MSU", "Big Ten"),
    "Michigan State Spartans": ("Michigan State", "MSU", "Big Ten"),
    "Neb":                     ("Nebraska", "NEB", "Big Ten"),
    "Nebraska":                ("Nebraska", "NEB", "Big Ten"),
    "Nebraska Cornhuskers":    ("Nebraska", "NEB", "Big Ten"),
    "NW":                      ("Northwestern", "NW", "Big Ten"),
    "Northwestern":            ("Northwestern", "NW", "Big Ten"),
    "Northwestern Wildcats":   ("Northwestern", "NW", "Big Ten"),
    "Oregon":                  ("Oregon", "ORE", "Big Ten"),
    "Oregon Ducks":            ("Oregon", "ORE", "Big Ten"),
    "Ohio St":                 ("Ohio State", "OSU", "Big Ten"),
    "Ohio St.":                ("Ohio State", "OSU", "Big Ten"),
    "Ohio State":              ("Ohio State", "OSU", "Big Ten"),
    "Ohio State Buckeyes":     ("Ohio State", "OSU", "Big Ten"),
    "tOSU":                    ("Ohio State", "OSU", "Big Ten"),
    "PSU":                     ("Penn State", "PSU", "Big Ten"),
    "Penn St":                 ("Penn State", "PSU", "Big Ten"),
    "Penn St.":                ("Penn State", "PSU", "Big Ten"),
    "Penn State":              ("Penn State", "PSU", "Big Ten"),
    "Penn State Nittany Lions": ("Penn State", "PSU", "Big Ten"),
    "PUR":                     ("Purdue", "PUR", "Big Ten"),
    "Purdue":                  ("Purdue", "PUR", "Big Ten"),
    "Purdue Boilermakers":     ("Purdue", "PUR", "Big Ten"),
    "RU":                      ("Rutgers", "RUTG", "Big Ten"),
    "Rutgers":                 ("Rutgers", "RUTG", "Big Ten"),
    "Rutgers Scarlet Knights": ("Rutgers", "RUTG", "Big Ten"),
    "UCLA":                    ("UCLA", "UCLA", "Big Ten"),
    "UCLA Bruins":             ("UCLA", "UCLA", "Big Ten"),
    "Southern California":     ("Southern California", "USC", "Big Ten"),
    "USC":                     ("Southern California", "USC", "Big Ten"),
    "USC Trojans":             ("Southern California", "USC", "Big Ten"),
    "Washington":              ("Washington", "WASH", "Big Ten"),
    "Washington (WA)":         ("Washington", "WASH", "Big Ten"),
    "Washington Huskies":      ("Washington", "WASH", "Big Ten"),
    "Wisc":                    ("Wisconsin", "WIS", "Big Ten"),
    "Wisc.":                   ("Wisconsin", "WIS", "Big Ten"),
    "Wisconsin":               ("Wisconsin", "WIS", "Big Ten"),
    "Wisconsin Badgers":       ("Wisconsin", "WIS", "Big Ten"),

    # ── CUSA ─────────────────────────────────────────────────────────────
    "Delaware Blue Hens":      ("Delaware", "DELA", "CUSA"),
    "FIU":                     ("Florida International", "FIU", "CUSA"),
    "Florida International":   ("Florida International", "FIU", "CUSA"),
    "Florida International Panthers": ("Florida International", "FIU", "CUSA"),
    "JSU":                     ("Jacksonville State", "JSU", "CUSA"),
    "Jacksonville State":      ("Jacksonville State", "JSU", "CUSA"),
    "Jacksonville State Gamecocks": ("Jacksonville State", "JSU", "CUSA"),
    "Kennesaw State":          ("Kennesaw State", "KSU", "CUSA"),
    "Kennesaw State Owls":     ("Kennesaw State", "KSU", "CUSA"),
    "Liberty":                 ("Liberty", "LIB", "CUSA"),
    "Liberty Flames":          ("Liberty", "LIB", "CUSA"),
    "Louisiana Tech":          ("Louisiana Tech", "LT", "CUSA"),
    "Louisiana Tech Bulldogs": ("Louisiana Tech", "LT", "CUSA"),
    "Missouri State Bears":    ("Missouri State", "MST", "CUSA"),
    "MTSU":                    ("Middle Tennessee State", "MTSU", "CUSA"),
    "Middle Tennessee":        ("Middle Tennessee State", "MTSU", "CUSA"),
    "Middle Tennessee Blue Raiders": ("Middle Tennessee State", "MTSU", "CUSA"),
    "NMSU":                    ("New Mexico State", "NMSU", "CUSA"),
    "New Mexico State":        ("New Mexico State", "NMSU", "CUSA"),
    "New Mexico State Aggies": ("New Mexico State", "NMSU", "CUSA"),
    "Sam Houston":             ("Sam Houston State", "SHSU", "CUSA"),
    "Sam Houston Bearkats":    ("Sam Houston State", "SHSU", "CUSA"),
    "Sam Houston State":       ("Sam Houston State", "SHSU", "CUSA"),
    "UTEP":                    ("UT El Paso", "UTEP", "CUSA"),
    "UTEP Miners":             ("UT El Paso", "UTEP", "CUSA"),
    "WKU":                     ("Western Kentucky", "WKU", "CUSA"),
    "Western Kentucky":        ("Western Kentucky", "WKU", "CUSA"),
    "Western Kentucky Hilltoppers": ("Western Kentucky", "WKU", "CUSA"),

    # ── Independent ──────────────────────────────────────────────────────
    "Connecticut":             ("Connecticut", "CONN", "Independent"),
    "UConn":                   ("Connecticut", "CONN", "Independent"),
    "UConn Huskies":           ("Connecticut", "CONN", "Independent"),
    "Massachusetts":           ("Massachusetts", "MASS", "Independent"),
    "UMass":                   ("Massachusetts", "MASS", "Independent"),
    "ND":                      ("Notre Dame", "ND", "Independent"),
    "Notre Dame":              ("Notre Dame", "ND", "Independent"),
    "Notre Dame Fighting Irish": ("Notre Dame", "ND", "Independent"),

# ── MAC ──────────────────────────────────────────────────────────────
    "Akron":                   ("Akron", "AKR", "MAC"),
    "Akron Zips":              ("Akron", "AKR", "MAC"),
    "Ball State":              ("Ball State", "BALL", "MAC"),
    "Ball State Cardinals":    ("Ball State", "BALL", "MAC"),
    "Bowling Green":           ("Bowling Green", "BGSU", "MAC"),
    "Bowling Green Falcons":   ("Bowling Green", "BGSU", "MAC"),
    "Buffalo":                 ("Buffalo", "BUFF", "MAC"),
    "Buffalo Bulls":           ("Buffalo", "BUFF", "MAC"),
    "CMU":                     ("Central Michigan", "CMU", "MAC"),
    "Central Michigan":        ("Central Michigan", "CMU", "MAC"),
    "Central Michigan Chippewas": ("Central Michigan", "CMU", "MAC"),
    "Eastern Michigan":        ("Eastern Michigan", "EMU", "MAC"),
    "Eastern Michigan Eagles": ("Eastern Michigan", "EMU", "MAC"),
    "Kent State":              ("Kent State", "KENT", "MAC"),
    "Kent State Golden Flashes": ("Kent State", "KENT", "MAC"),
    "Massachusetts Minutemen": ("Massachusetts", "MASS", "MAC"),
    "Miami (OH)":              ("Miami (OH)", "MOH", "MAC"),
    "Miami (OH) RedHawks":     ("Miami (OH)", "MOH", "MAC"),
    "Miami OH":                ("Miami (OH)", "MOH", "MAC"),
    "NIU":                     ("Northern Illinois", "NIU", "MAC"),
    "Northern Illinois":       ("Northern Illinois", "NIU", "MAC"),
    "Northern Illinois Huskies": ("Northern Illinois", "NIU", "MAC"),
    "Ohio":                    ("Ohio", "OHIO", "MAC"),
    "Ohio Bobcats":            ("Ohio", "OHIO", "MAC"),
    "Toledo":                  ("Toledo", "TOL", "MAC"),
    "Toledo Rockets":          ("Toledo", "TOL", "MAC"),
    "WMU":                     ("Western Michigan", "WMU", "MAC"),
    "Western Michigan":        ("Western Michigan", "WMU", "MAC"),
    "Western Michigan Broncos": ("Western Michigan", "WMU", "MAC"),

    # ── MVFC ─────────────────────────────────────────────────────────────
    "NDSU":                    ("North Dakota State", "NDSU", "MVFC"),
    "North Dakota St.":        ("North Dakota State", "NDSU", "MVFC"),
    "North Dakota State":      ("North Dakota State", "NDSU", "MVFC"),
    "North Dakota Fighting Hawks": ("North Dakota", "UND", "MVFC"),
    "SDST":                    ("South Dakota State", "SDSU", "MVFC"),    
    "S. Dakota State":         ("South Dakota State", "SDSU", "MVFC"),
    "South Dakota St":         ("South Dakota State", "SDSU", "MVFC"),
    "South Dakota St.":        ("South Dakota State", "SDSU", "MVFC"),
    "South Dakota State":      ("South Dakota State", "SDSU", "MVFC"),
    "North Dakota State Bison": ("North Dakota State", "NDSU", "MVFC"),
    "South Dakota State Jackrabbits": ("South Dakota State", "SDSU", "MVFC"),    
    "SDAK":                    ("South Dakota", "SDAK", "MVFC"),
    "South Dakota Coyotes":    ("South Dakota", "SDAK", "MVFC"),
    "UND":                     ("North Dakota", "UND", "MVFC"),
    "YSU":                     ("Youngstown State", "YSU", "MVFC"),
    "Youngstown State Penguins": ("Youngstown State", "YSU", "MVFC"),
    "ILST":                    ("Illinois State", "ILST", "MVFC"),
    "Illinois State Redbirds": ("Illinois State", "ILST", "MVFC"),
    "SIU":                     ("Southern Illinois", "SIU", "MVFC"),
    "Southern Illinois Salukis": ("Southern Illinois", "SIU", "MVFC"),
    "UNI":                     ("Northern Iowa", "UNI", "MVFC"),
    "Northern Iowa Panthers":  ("Northern Iowa", "UNI", "MVFC"),
    "MUR":                     ("Murray State", "MUR", "MVFC"),
    "Murray State Racers":     ("Murray State", "MUR", "MVFC"),
    "INST":                    ("Indiana State", "INST", "MVFC"),
    "Indiana State Sycamores": ("Indiana State", "INST", "MVFC"),

# ── Mountain West ────────────────────────────────────────────────────
    "AFA":                    ("Air Force", "AFA", "Mountain West"),
    "Air Force":              ("Air Force", "AFA", "Mountain West"),
    "Air Force Falcons":      ("Air Force", "AFA", "Mountain West"),
    "BOIS":                   ("Boise State", "BSU", "Mountain West"),
    "BSU":                    ("Boise State", "BSU", "Mountain West"),
    "Boise St.":              ("Boise State", "BSU", "Mountain West"),
    "Boise State":            ("Boise State", "BSU", "Mountain West"),
    "Boise State Broncos":    ("Boise State", "BSU", "Mountain West"),
    "CSU":                    ("Colorado State", "CSU", "Mountain West"),
    "Colo St.":               ("Colorado State", "CSU", "Mountain West"),
    "Colo State":             ("Colorado State", "CSU", "Mountain West"),
    "Colorado State":         ("Colorado State", "CSU", "Mountain West"),
    "Colorado State Rams":    ("Colorado State", "CSU", "Mountain West"),
    "FRES":                   ("Fresno State", "FRES", "Mountain West"),
    "Fresno St.":             ("Fresno State", "FRES", "Mountain West"),
    "Fresno State":           ("Fresno State", "FRES", "Mountain West"),
    "Fresno State Bulldogs":  ("Fresno State", "FRES", "Mountain West"),
    "HAW":                    ("Hawaii", "HAW", "Mountain West"),
    "Hawaii":                 ("Hawaii", "HAW", "Mountain West"),
    "Hawai'i Rainbow Warriors": ("Hawaii", "HAW", "Mountain West"),
    "NEV":                    ("Nevada", "NEV", "Mountain West"),
    "Nevada":                 ("Nevada", "NEV", "Mountain West"),
    "Nevada Wolf Pack":       ("Nevada", "NEV", "Mountain West"),
    "Nevada-Las Vegas":       ("Nevada-Las Vegas", "UNLV", "Mountain West"),
    "UNM":                    ("New Mexico", "UNM", "Mountain West"),
    "New Mexico":             ("New Mexico", "UNM", "Mountain West"),
    "New Mexico Lobos":       ("New Mexico", "UNM", "Mountain West"),
    "SDSU":                   ("San Diego State", "SDSU", "Mountain West"),
    "San Diego State":        ("San Diego State", "SDSU", "Mountain West"),
    "San Diego State Aztecs": ("San Diego State", "SDSU", "Mountain West"),
    "SJSU":                   ("San José State", "SJSU", "Mountain West"),
    "San Jose State":         ("San Jose State", "SJSU", "Mountain West"),
    "San José State Spartans": ("San José State", "SJSU", "Mountain West"),
    "UNLV":                   ("Nevada-Las Vegas", "UNLV", "Mountain West"),
    "UNLV Rebels":            ("Nevada-Las Vegas", "UNLV", "Mountain West"),
    "USU":                    ("Utah State", "USU", "Mountain West"),
    "Utah St.":               ("Utah State", "USU", "Mountain West"),
    "Utah State":             ("Utah State", "USU", "Mountain West"),
    "Utah State Aggies":      ("Utah State", "USU", "Mountain West"),
    "Wyo":                    ("Wyoming", "WYO", "Mountain West"),
    "Wyoming":                ("Wyoming", "WYO", "Mountain West"),
    "Wyoming Cowboys":        ("Wyoming", "WYO", "Mountain West"),

    # ── Pac-12 ───────────────────────────────────────────────────────────
    "ORST":                    ("Oregon State", "ORST", "Pac-12"),
    "Ore St":                  ("Oregon State", "ORST", "Pac-12"),
    "Ore St.":                 ("Oregon State", "ORST", "Pac-12"),
    "Ore State":               ("Oregon State", "ORST", "Pac-12"),
    "Oregon State":            ("Oregon State", "ORST", "Pac-12"),
    "Oregon State Beavers":    ("Oregon State", "ORST", "Pac-12"),
    "WSU":                     ("Washington State", "WSU", "Pac-12"),
    "Wash St":                 ("Washington State", "WSU", "Pac-12"),
    "Wash St.":                ("Washington State", "WSU", "Pac-12"),
    "Wash State":              ("Washington State", "WSU", "Pac-12"),
    "Washington State":        ("Washington State", "WSU", "Pac-12"),
    "Washington State Cougars": ("Washington State", "WSU", "Pac-12"),
    "Wazzu":                   ("Washington State", "WSU", "Pac-12"),

    # ── SEC ──────────────────────────────────────────────────────────────
    "ALA":                     ("Alabama", "ALA", "SEC"),
    "Alabama":                 ("Alabama", "ALA", "SEC"),
    "Alabama Crimson Tide":    ("Alabama", "ALA", "SEC"),
    "Bama":                    ("Alabama", "ALA", "SEC"),
    "Ark":                     ("Arkansas", "ARK", "SEC"),
    "Ark.":                    ("Arkansas", "ARK", "SEC"),
    "Arkansas":                ("Arkansas", "ARK", "SEC"),
    "Arkansas Razorbacks":     ("Arkansas", "ARK", "SEC"),
    "AUB":                     ("Auburn", "AUB", "SEC"),
    "Auburn":                  ("Auburn", "AUB", "SEC"),
    "Auburn Tigers":           ("Auburn", "AUB", "SEC"),
    "FLA":                     ("Florida", "FLA", "SEC"),
    "Florida":                 ("Florida", "FLA", "SEC"),
    "Florida Gators":          ("Florida", "FLA", "SEC"),
    "UF":                      ("Florida", "FLA", "SEC"),
    "LSU":                     ("Louisiana State", "LSU", "SEC"),
    "LSU Tigers":              ("Louisiana State", "LSU", "SEC"),
    "Louisiana State":         ("Louisiana State", "LSU", "SEC"),
    "MISS":                    ("Mississippi", "MISS", "SEC"),
    "Mississippi":             ("Mississippi", "MISS", "SEC"),
    "Ole Miss":                ("Mississippi", "MISS", "SEC"),
    "MIZ":                     ("Missouri", "MIZZ", "SEC"),
    "Missouri":                ("Missouri", "MIZZ", "SEC"),
    "Missouri Tigers":         ("Missouri", "MIZZ", "SEC"),
    "Mizzou":                  ("Missouri", "MIZZ", "SEC"),
    "MSST":                    ("Mississippi State", "MSST", "SEC"),
    "Miss St":                 ("Mississippi State", "MSST", "SEC"),
    "Mississippi St.":         ("Mississippi State", "MSST", "SEC"),
    "Mississippi State":       ("Mississippi State", "MSST", "SEC"),
    "Mississippi State Bulldogs": ("Mississippi State", "MSST", "SEC"),
    "OU":                      ("Oklahoma", "OKLA", "SEC"),
    "Oklahoma":                ("Oklahoma", "OKLA", "SEC"),
    "Oklahoma Sooners":        ("Oklahoma", "OKLA", "SEC"),
    "S. Carolina":             ("South Carolina", "SC", "SEC"),
    "SC":                      ("South Carolina", "SC", "SEC"),
    "South Carolina":          ("South Carolina", "SC", "SEC"),
    "South Carolina Gamecocks": ("South Carolina", "SC", "SEC"),
    "USC-East":                ("South Carolina", "SC", "SEC"),
    "TAMU":                    ("Texas A&M", "TAMU", "SEC"),
    "Texas A & M":             ("Texas A&M", "TAMU", "SEC"),
    "Texas A&M":               ("Texas A&M", "TAMU", "SEC"),
    "Texas A&M Aggies":        ("Texas A&M", "TAMU", "SEC"),
    "Texas AM":                ("Texas A&M", "TAMU", "SEC"),
    "Tenn":                    ("Tennessee", "UT-TN", "SEC"),
    "Tennessee":               ("Tennessee", "UT-TN", "SEC"),
    "Tennessee Volunteers":    ("Tennessee", "UT-TN", "SEC"),
    "UTK":                     ("Tennessee", "UT-TN", "SEC"),
    "TEX":                     ("Texas", "UT-TX", "SEC"),
    "Texas":                   ("Texas", "UT-TX", "SEC"),
    "Texas Longhorns":         ("Texas", "UT-TX", "SEC"),
    "UT":                      ("Texas", "UT-TX", "SEC"),
    "Georgia":                 ("Georgia", "UGA", "SEC"),
    "Georgia Bullsdogs":       ("Georgia", "UGA", "SEC"),
    "UGA":                     ("Georgia", "UGA", "SEC"),
    "Kentucky":                ("Kentucky", "UK", "SEC"),
    "Kentucky Wildcats":       ("Kentucky", "UK", "SEC"),
    "UK":                      ("Kentucky", "UK", "SEC"),
    "VAN":                     ("Vanderbilt", "VANDY", "SEC"),
    "Vanderbilt":              ("Vanderbilt", "VANDY", "SEC"),
    "Vanderbilt Commodores":   ("Vanderbilt", "VANDY", "SEC"),
    "Vandy":                   ("Vanderbilt", "VANDY", "SEC"),

    # ── Sun Belt ─────────────────────────────────────────────────────────
    "APP":                     ("Appalachian State", "APP", "Sun Belt"),
    "App St":                  ("Appalachian State", "APP", "Sun Belt"),
    "App State":               ("Appalachian State", "APP", "Sun Belt"),
    "App State Mountaineers":  ("Appalachian State", "APP", "Sun Belt"),
    "Appalachian State":       ("Appalachian State", "APP", "Sun Belt"),
    "ARST":                    ("Arkansas State", "ARST", "Sun Belt"),
    "Arkansas State":          ("Arkansas State", "ARST", "Sun Belt"),
    "CCU":                     ("Coastal Carolina", "CCU", "Sun Belt"),
    "Coastal Carolina":        ("Coastal Carolina", "CCU", "Sun Belt"),
    "GASO":                    ("Georgia Southern", "GASO", "Sun Belt"),
    "Georgia Southern":        ("Georgia Southern", "GASO", "Sun Belt"),
    "GAST":                    ("Georgia State", "GAST", "Sun Belt"),
    "Georgia State":           ("Georgia State", "GAST", "Sun Belt"),
    "JMU":                     ("James Madison", "JMU", "Sun Belt"),
    "James Madison":           ("James Madison", "JMU", "Sun Belt"),
    "MRSH":                    ("Marshall", "MARSH", "Sun Belt"),
    "Marshall":                ("Marshall", "MARSH", "Sun Belt"),
    "ODU":                     ("Old Dominion", "ODU", "Sun Belt"),
    "Old Dominion":            ("Old Dominion", "ODU", "Sun Belt"),
    "Troy":                    ("Troy", "TROY", "Sun Belt"),
    "Troy Trojans":            ("Troy", "TROY", "Sun Belt"),
    "TXST":                    ("Texas State", "TXST", "Sun Belt"),
    "Texas State":             ("Texas State", "TXST", "Sun Belt"),
    "Texas State Bobcats":     ("Texas State", "TXST", "Sun Belt"),
    "Louisiana":               ("Louisiana", "ULL", "Sun Belt"),
    "Louisiana Ragin' Cajuns": ("Louisiana", "ULL", "Sun Belt"),
    "UL":                      ("Louisiana", "ULL", "Sun Belt"),
    "Louisiana-Lafayette":     ("Louisiana-Lafayette", "ULL", "Sun Belt"),
    "ULL":                     ("Louisiana-Lafayette", "ULL", "Sun Belt"),
    "Louisiana-Monroe":        ("Louisiana-Monroe", "ULM", "Sun Belt"),
    "UL Monroe":               ("Louisiana-Monroe", "ULM", "Sun Belt"),
    "UL Monroe Warhawks":      ("Louisiana-Monroe", "ULM", "Sun Belt"),
    "ULM":                     ("Louisiana-Monroe", "ULM", "Sun Belt"),
    "South Alabama":           ("South Alabama", "USA", "Sun Belt"),
    "South Alabama Jaguars":   ("South Alabama", "USA", "Sun Belt"),
    "USA":                     ("South Alabama", "USA", "Sun Belt"),
    "Southern Miss":           ("Southern Miss", "USM", "Sun Belt"),
    "Southern Miss Golden Eagles": ("Southern Miss", "USM", "Sun Belt"),
    "USM":                     ("Southern Miss", "USM", "Sun Belt"),
}

dim_school_rows = [
    {"school_raw": k, "school_canonical": v[0], "school_abbr": v[1], "conference": v[2]}
    for k, v in _school_aliases.items()
]

# Create DataFrame
dim_school = pd.DataFrame(dim_school_rows)

# Save to Parquet
out_path = Path(CFG.data_dir) / "dim_school.parquet"
dim_school.to_parquet(out_path, index=False)

print(f"dim_school: {len(dim_school)} alias mappings saved to → {out_path}")

dim_school: 436 alias mappings saved to → C:\Users\benha\OneDrive\Documents\GitHub\Python-PowerBI-DynastyFantasyFootball\data\dim_school.parquet


## 3 — Helper Functions

Reusable parsing/cleaning functions that replace the M code transformations.

In [5]:
# Player helpers are imported from etl_helpers (single source of truth).
import sys
from pathlib import Path
for _p in (Path.cwd() / "notebooks", Path.cwd()):
    if (_p / "etl_helpers.py").exists():
        sys.path.insert(0, str(_p)); break
from etl_helpers import parse_height_to_inches, clean_player_name, generate_player_key

assert parse_height_to_inches("6'2") == 74.0
assert clean_player_name("J.K. Dobbins") == "jk dobbins"
assert clean_player_name("D'Andre Swift") == "d'andre swift"
print("Helpers OK (etl_helpers)")

Helper functions: all smoke tests passed


## 4 -- Pull nflverse Combine Data (Prospect Seed)

Foundation of `dim_rookie_prospect`. Every combine invitee for `CFG.draft_year`
gets a row, even if they opted out of all drills.

In [6]:
# ── Pull combine data via nflreadpy ─────────────────────────────────────────
# nflreadpy returns Polars DataFrames; convert to pandas immediately
print("Loading combine data via nflreadpy...")
try:
    combine_all = nfl.load_combine(seasons=True).to_pandas()
except Exception as e:
    raise RuntimeError(f"Failed to load nflverse combine data: {e}") from e

print(f"Total combine records (all years): {len(combine_all):,}")
print(f"Columns: {list(combine_all.columns)}")
print(f"Year range: {combine_all['season'].min()} – {combine_all['season'].max()}")
print(f"{CFG.draft_year} invitees: {len(combine_all[combine_all['season'] == CFG.draft_year])}")

combine_current = combine_all[combine_all["season"] == CFG.draft_year].copy()
combine_current.head(5)

Loading combine data via nflreadpy...
Total combine records (all years): 8,968
Columns: ['season', 'draft_year', 'draft_team', 'draft_round', 'draft_ovr', 'pfr_id', 'cfb_id', 'player_name', 'pos', 'school', 'ht', 'wt', 'forty', 'bench', 'vertical', 'broad_jump', 'cone', 'shuttle']
Year range: 2000 – 2026
2026 invitees: 319


,season,draft_year,draft_team,draft_round,draft_ovr,pfr_id,cfb_id,player_name,pos,school,ht,wt,forty,bench,vertical,broad_jump,cone,shuttle
8649,2026,NaN,NaN,NaN,NaN,AdamCh01,chris-adams-2,Chris Adams,OT,Memphis,6-5,311.0,NaN,NaN,NaN,NaN,NaN,NaN
8650,2026,NaN,NaN,NaN,NaN,NaN,joey-aguilar-1,Jose Aguilar,QB,Tennessee,6-3,229.0,NaN,NaN,NaN,NaN,NaN,NaN
8651,2026,NaN,NaN,NaN,NaN,AllaDr00,drew-allar-1,Drew Allar,QB,Penn St.,6-5,228.0,NaN,NaN,NaN,NaN,NaN,NaN
8652,2026,NaN,NaN,NaN,NaN,NaN,cj-allen-1,CJ Allen,LB,Georgia,6-1,230.0,NaN,NaN,NaN,NaN,NaN,NaN
8653,2026,NaN,NaN,NaN,NaN,AlleKa01,kaytron-allen-1,Kaytron Allen,RB,Penn St.,5-11,216.0,NaN,NaN,NaN,NaN,NaN,NaN


## 5 -- Build dim_rookie_prospect from Combine Seed

One row per combine invitee. `player_key` is the interim PK (MD5 hash of
name + position + school) used across fact tables pre-draft.
`gsis_id` is null at seed time; populated post-signing via ETL join against
`dim_nfl_players` on `pfr_id`.

In [7]:
def build_dim_rookie_prospect(combine_df: pd.DataFrame, draft_year: int) -> pd.DataFrame:
    # Filter to current draft year only
    df = combine_df[combine_df["season"] == draft_year].copy()

    if len(df) == 0:
        print(f"Warning: No combine data for {draft_year}. Returning empty frame.")
        return pd.DataFrame()

    # -- Load transformer tables ------------------------------------------
    pos_map    = pd.read_parquet(DATA / "dim_position.parquet")
    school_map = pd.read_parquet(DATA / "dim_school.parquet")

    # -- Normalize raw fields ---------------------------------------------
    df["player_name"]       = df["player_name"].str.strip()
    df["player_name_clean"] = df["player_name"].apply(clean_player_name)
    df["position_raw"]      = df["pos"].str.upper().str.strip()
    df["school_raw"]        = df["school"].str.strip()

    # -- Position lookup --------------------------------------------------
    df = df.merge(
        pos_map[["position_raw", "position_detail", "position_group",
                 "side_of_ball", "fantasy_relevant"]],
        on="position_raw", how="left"
    )
    unmatched = df[df["position_detail"].isna()]["position_raw"].unique()
    if len(unmatched):
        print(f"Warning: unmatched positions (add to dim_position): {list(unmatched)}")

    # -- School lookup ----------------------------------------------------
    df = df.merge(
        school_map.rename(columns={"school_raw": "school_raw_lookup"}),
        left_on="school_raw", right_on="school_raw_lookup", how="left"
    )
    df["school_canonical"] = df["school_canonical"].fillna(df["school_raw"])
    df["conference"]       = df["conference"].fillna("Unknown")
    df.drop(columns=["school_raw_lookup"], inplace=True, errors="ignore")

    # -- Height -----------------------------------------------------------
    df["height_inches"] = df["ht"].apply(parse_height_to_inches)

    # -- Interim player key (MD5 hash of name + position + school) --------
    df["player_key"] = df.apply(
        lambda r: generate_player_key(r["player_name"], r["position_raw"], r["school_raw"]),
        axis=1
    )

    # -- Select final columns ---------------------------------------------
    result = df[[
        "player_key",
        "player_name", "player_name_clean",
        "position_raw", "position_detail", "position_group",
        "side_of_ball", "fantasy_relevant",
        "school_raw", "school_canonical", "conference",
        "height_inches", "wt",
        "pfr_id", "cfb_id",
    ]].copy()
    result.rename(columns={"wt": "weight"}, inplace=True)

    # gsis_id: null at seed time; ETL populates after rookie signing
    # via pfr_id join against dim_nfl_players
    result["gsis_id"]    = pd.NA
    result["draft_year"] = draft_year
    result["source"]     = "nflverse_combine"
    result["added_date"] = TODAY

    result.drop_duplicates(subset=["player_key"], inplace=True)
    return result.reset_index(drop=True)


dim_rookie_prospect = build_dim_rookie_prospect(combine_all, CFG.draft_year)
print(f"dim_rookie_prospect: {len(dim_rookie_prospect)} prospects for {CFG.draft_year}")
print(f"Position groups:\n{dim_rookie_prospect['position_group'].value_counts().to_string()}")
print(f"Fantasy-relevant: {dim_rookie_prospect['fantasy_relevant'].sum()}")

out_path = DATA / "dim_rookie_prospect.parquet"
dim_rookie_prospect.to_parquet(out_path, index=False)
print(f"Saved -> {out_path}")
dim_rookie_prospect.head(10)

dim_rookie_prospect: 319 prospects for 2026
Position groups:
position_group
DL    64
OL    56
DB    54
WR    46
LB    28
TE    27
RB    21
QB    16
ST     7
Fantasy-relevant: 255
Saved -> data\dim_rookie_prospect.parquet


,player_key,player_name,player_name_clean,position_raw,position_detail,position_group,side_of_ball,fantasy_relevant,school_raw,school_canonical,conference,height_inches,weight,pfr_id,cfb_id,gsis_id,draft_year,source,added_date
0,4f61e36da0f6,Chris Adams,chris adams,OT,OT,OL,Offense,False,Memphis,Memphis,Unknown,77.0,311.0,AdamCh01,chris-adams-2,<NA>,2026,nflverse_combine,2026-05-16
1,a912fb2018e4,Jose Aguilar,jose aguilar,QB,QB,QB,Offense,True,Tennessee,Tennessee,Unknown,75.0,229.0,NaN,joey-aguilar-1,<NA>,2026,nflverse_combine,2026-05-16
2,e5417bc86706,Drew Allar,drew allar,QB,QB,QB,Offense,True,Penn St.,Penn State,Big Ten,77.0,228.0,AllaDr00,drew-allar-1,<NA>,2026,nflverse_combine,2026-05-16
3,55c56b655c49,CJ Allen,cj allen,LB,LB,LB,Defense,True,Georgia,Georgia,SEC,73.0,230.0,NaN,cj-allen-1,<NA>,2026,nflverse_combine,2026-05-16
4,85fc4414cd8f,Kaytron Allen,kaytron allen,RB,RB,RB,Offense,True,Penn St.,Penn State,Big Ten,71.0,216.0,AlleKa01,kaytron-allen-1,<NA>,2026,nflverse_combine,2026-05-16
5,3df670e6c8ad,Marcus Allen,marcus allen,CB,CB,DB,Defense,True,North Carolina,North Carolina,ACC,74.0,187.0,AlleMa05,marcus-allen-6,<NA>,2026,nflverse_combine,2026-05-16
6,0fb64b35d726,Luke Altmyer,luke altmyer,QB,QB,QB,Offense,True,Illinois,Illinois,Unknown,74.0,210.0,AltmLu00,luke-altmyer-1,<NA>,2026,nflverse_combine,2026-05-16
7,173d5dc9a360,Aaron Anderson,aaron anderson,WR,WR,WR,Offense,True,LSU,Louisiana State,SEC,68.0,191.0,AndeAa00,aaron-anderson-3,<NA>,2026,nflverse_combine,2026-05-16
8,e025966653e5,David Bailey,david bailey,EDGE,EDGE,DL,Defense,True,Texas Tech,Texas Tech,Unknown,76.0,251.0,BailDa02,david-bailey-5,<NA>,2026,nflverse_combine,2026-05-16
9,f35fadfc88cc,Rueben Bain,rueben bain,EDGE,EDGE,DL,Defense,True,Miami,Miami (FL),ACC,74.0,263.0,BainRu00,rueben-bain-jr-1,<NA>,2026,nflverse_combine,2026-05-16


## 6 -- Validation & Summary

In [8]:
dim_rookie_prospect = pd.read_parquet(DATA / "dim_rookie_prospect.parquet")

print(f"dim_rookie_prospect: {len(dim_rookie_prospect)} prospects for {CFG.draft_year}")
print(f"\nBy position group:")
print(dim_rookie_prospect["position_group"].value_counts().to_string())
print(f"\nBy source:")
print(dim_rookie_prospect["source"].value_counts().to_string())
print(f"\nTop 10 schools:")
print(dim_rookie_prospect["school_canonical"].value_counts().head(10).to_string())
print(f"Missing height : {dim_rookie_prospect['height_inches'].isna().sum()}")
print(f"Missing weight : {dim_rookie_prospect['weight'].isna().sum()}")
print(f"gsis_id status : all null at seed time = {dim_rookie_prospect['gsis_id'].isna().all()}")

# Unmatched positions
unmatched = dim_rookie_prospect[dim_rookie_prospect["position_detail"].isna()]
if len(unmatched):
    print(f"\nWARN: {len(unmatched)} players with unmatched positions:")
    print(unmatched[["player_name", "position_raw"]].to_string())
else:
    print("\nOK: all positions resolved")

dim_rookie_prospect: 319 prospects for 2026

By position group:
position_group
DL    64
OL    56
DB    54
WR    46
LB    28
TE    27
RB    21
QB    16
ST     7

By source:
source
nflverse_combine    319

Top 10 schools:
school_canonical
Texas A&M          13
Alabama            12
Louisiana State    11
Ohio State         11
Georgia            10
Miami (FL)         10
Oklahoma           10
Penn State          9
Florida             9
Clemson             9
Missing height : 0
Missing weight : 0
gsis_id status : all null at seed time = True

OK: all positions resolved
